Setting up Model: Qwen 2.5-7B-Instruct

In [ ]:
!nvidia-smi

Fri Oct 30 19:24:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P0             27W /   70W |    8291MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.39.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.4.0 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 2.4.0 which is incompatible.
gradio-client 2.5.0 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.4.0 which is incompatible.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16
)

print("Model loaded successfully!")

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
messages = [
    {
        "role": "user",
        "content": "You are an AI agent. What are the basic steps for searching for an object in a room?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to("cuda")

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

Searching for an object in a room can be approached systematically to ensure that you efficiently locate what you're looking for. Here are some basic steps you can follow:

1. **Define the Object**: Clearly identify what you are looking for. Knowing the exact item and its characteristics (size, color, shape, etc.) will help narrow down your search.

2. **Plan Your Search**: Before starting, think about where the object might be located. Consider common places where such items are typically stored or used. For example, books are often found on shelves, keys might be in a key holder near the entrance, and documents could be in a filing cabinet or on a desk.

3. **Start with Likely Locations**: Begin your search in areas where the


Connecting to benchmark (ALFWorld)

In [ ]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [ ]:
%pip install -q alfworld

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.3/660.3 kB 29.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.3/356.3 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.5/101.5 kB 12.0 MB/s eta 0:00:00


In [ ]:
!alfworld-download

Extracting json_2.1.1_json.zip: 100% 7302/7302 [00:06<00:00, 1061.45it/s]
Extracting json_2.1.1_pddl.zip: 100% 7080/7080 [00:02<00:00, 3137.47it/s]
Extracting json_2.1.2_tw-pddl.zip: 100% 4027/4027 [00:01<00:00, 2038.82it/s]


In [ ]:
!wget -q https://raw.githubusercontent.com/alfworld/alfworld/master/configs/base_config.yaml -O /content/alfworld_config.yaml

In [ ]:
import os
import yaml
from alfworld.agents.environment import get_environment

os.environ["ALFWORLD_DATA"] = os.path.expanduser(
    "~/.cache/alfworld"
)

with open("/content/alfworld_config.yaml") as f:
    config = yaml.safe_load(f)

config["dataset"]["num_train_games"] = 1

env = get_environment("AlfredTWEnv")(
    config,
    train_eval="train"
).init_env(batch_size=1)

obs, info = env.reset()

print("INITIAL OBSERVATION:")
print(obs[0])

actions = list(info["admissible_commands"][0])

print("\nAVAILABLE ACTIONS:")
print(actions[:15])

action = "look" if "look" in actions else actions[0]

next_obs, scores, dones, next_info = env.step([action])

print("\nACTION EXECUTED:", action)
print("\nNEW OBSERVATION:")
print(next_obs[0])
print("\nDONE:", dones[0])

Initializing AlfredTWEnv...


100%|██████████| 8810/8810 [00:07<00:00, 1156.16it/s]


Overall we have 3553 games in split=train
Training with 1 games
INITIAL OBSERVATION:
-= Welcome to TextWorld, ALFRED! =-

You are in the middle of a room. Looking quickly around you, you see a cabinet 21, a cabinet 20, a cabinet 19, a cabinet 18, a cabinet 17, a cabinet 16, a cabinet 15, a cabinet 14, a cabinet 13, a cabinet 12, a cabinet 11, a cabinet 10, a cabinet 9, a cabinet 8, a cabinet 7, a cabinet 6, a cabinet 5, a cabinet 4, a cabinet 3, a cabinet 2, a cabinet 1, a coffeemachine 1, a countertop 2, a countertop 1, a diningtable 1, a drawer 5, a drawer 4, a drawer 3, a drawer 2, a drawer 1, a fridge 1, a garbagecan 1, a microwave 1, a sinkbasin 1, a stoveburner 4, a stoveburner 3, a stoveburner 2, a stoveburner 1, and a toaster 1.

Your task is to: cool some apple and put it in garbagecan.

AVAILABLE ACTIONS:
['go to cabinet 1', 'go to cabinet 10', 'go to cabinet 11', 'go to cabinet 12', 'go to cabinet 13', 'go to cabinet 14', 'go to cabinet 15', 'go to cabinet 16', 'go to cabine

In [ ]:
print("Model:", "model" in globals())
print("Tokenizer:", "tokenizer" in globals())
print("ALFWorld:", "env" in globals())

Model: True
Tokenizer: True
ALFWorld: False


In [ ]:
%pip install -q -U "bitsandbytes>=0.46.1" accelerate

In [ ]:
import importlib.metadata
import torch

print("bitsandbytes:", importlib.metadata.version("bitsandbytes"))
print("CUDA available:", torch.cuda.is_available())

bitsandbytes: 0.50.2
CUDA available: True


RESETTING AFTER SOME ERRORS

In [ ]:
import torch
import bitsandbytes as bnb
import transformers
from transformers.utils import is_bitsandbytes_available

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())
print("bitsandbytes available:", is_bitsandbytes_available())

PyTorch: 2.11.0+cu128
Transformers: 5.22.1
bitsandbytes: 0.50.2
CUDA available: True
bitsandbytes available: True


In [ ]:
import os
import yaml
from alfworld.agents.environment import get_environment

os.environ["ALFWORLD_DATA"] = os.path.expanduser(
    "~/.cache/alfworld"
)

with open("/content/alfworld_config.yaml") as f:
    config = yaml.safe_load(f)

config["dataset"]["num_train_games"] = 1

env = get_environment("AlfredTWEnv")(
    config,
    train_eval="train"
).init_env(batch_size=1)

print("ALFWorld initialized successfully!")

Initializing AlfredTWEnv...


100%|██████████| 8810/8810 [00:17<00:00, 514.72it/s]

Overall we have 3553 games in split=train
Training with 1 games
ALFWorld initialized successfully!


In [ ]:
import torch
import re

# Start a benchmark task
obs, info = env.reset()

observation = obs[0]
actions = list(info["admissible_commands"][0])

print("TASK AND ENVIRONMENT:")
print(observation)

# Number each available action
action_list = "\n".join(
    f"{i+1}. {action}"
    for i, action in enumerate(actions)
)

# Construct prompt
messages = [
    {
        "role": "system",
        "content": (
            "You are an AI agent operating in ALFWorld. "
            "Your goal is to complete the assigned task. "
            "Choose one action from the available actions. "
            "Respond with ONLY the action number."
        )
    },
    {
        "role": "user",
        "content": (
            f"Current observation:\n{observation}\n\n"
            f"Available actions:\n{action_list}\n\n"
            "What is the best next action?"
        )
    }
]

# Tokenize
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to("cuda")

# Generate action
model.eval()

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
).strip()

print("\nRAW MODEL RESPONSE:", repr(response))

# Validate the output
match = re.fullmatch(r"\d+", response)

if match is None:
    raise ValueError(
        f"Invalid action number output: {response!r}"
    )

index = int(response) - 1

if not 0 <= index < len(actions):
    raise ValueError("Action number is out of range")

chosen_action = actions[index]

print("\nAGENT CHOSE:", chosen_action)

# Execute chosen action
next_obs, scores, dones, next_info = env.step(
    [chosen_action]
)

print("\nENVIRONMENT RESPONSE:")
print(next_obs[0])

print("\nTASK FINISHED:", dones[0])
print("TASK SUCCESS:", next_info["won"][0])

TASK AND ENVIRONMENT:
-= Welcome to TextWorld, ALFRED! =-

You are in the middle of a room. Looking quickly around you, you see a cabinet 21, a cabinet 20, a cabinet 19, a cabinet 18, a cabinet 17, a cabinet 16, a cabinet 15, a cabinet 14, a cabinet 13, a cabinet 12, a cabinet 11, a cabinet 10, a cabinet 9, a cabinet 8, a cabinet 7, a cabinet 6, a cabinet 5, a cabinet 4, a cabinet 3, a cabinet 2, a cabinet 1, a coffeemachine 1, a countertop 2, a countertop 1, a diningtable 1, a drawer 5, a drawer 4, a drawer 3, a drawer 2, a drawer 1, a fridge 1, a garbagecan 1, a microwave 1, a sinkbasin 1, a stoveburner 4, a stoveburner 3, a stoveburner 2, a stoveburner 1, and a toaster 1.

Your task is to: cool some apple and put it in garbagecan.

RAW MODEL RESPONSE: '1'

AGENT CHOSE: go to cabinet 1

ENVIRONMENT RESPONSE:
You arrive at cabinet 1. On the cabinet 1, you see nothing.

TASK FINISHED: False
TASK SUCCESS: False


RUNNING COMPLETE AGENT TRAJECTORY

In [ ]:
import torch
import re
import json

MAX_STEPS = 30
HISTORY_WINDOW = 8

# Reset environment
obs, info = env.reset()
observation = obs[0]
initial_observation = observation

# Extract the task so the agent never forgets it
task_match = re.search(
    r"Your task is to:\s*([^\n]+)",
    observation,
    flags=re.IGNORECASE
)

if not task_match:
    raise ValueError("Could not extract the task from ALFWorld")

task = task_match.group(1).strip()

history = []
trace = []
status = "max_steps"
model.eval()

print("=" * 60)
print("TASK:", task)
print("=" * 60)

for step in range(1, MAX_STEPS + 1):

    actions = list(info["admissible_commands"][0])

    if not actions:
        status = "no_actions"
        break

    # Number available actions
    action_list = "\n".join(
        f"{i+1}. {action}"
        for i, action in enumerate(actions)
    )

    # Preserve recent interactions
    history_text = "\n".join(
        f"Action: {item['action']}\n"
        f"Observation: {item['observation']}"
        for item in history
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are an intelligent agent operating in ALFWorld. "
                "Your objective is to complete the assigned household task. "

                "Before choosing an action, consider the sequence of "
                "subgoals required to complete the entire task. "

                "If a task requires modifying an object, such as "
                "heating, cooling, or cleaning, complete that "
                "modification before placing it at its destination. "

                "Use previous observations to remember object locations "
                "and which steps have already been completed. "

                "Avoid repeating actions that have not produced progress. "
                "Choose exactly one action from the numbered list. "
                "Respond ONLY with its number."
            )
        },

        {
            "role": "user",
            "content": (
                f"Task: {task}\n\n"
                f"Previous interactions:\n{history_text}\n\n"
                f"Current observation:\n{observation}\n\n"
                f"Available actions:\n{action_list}\n\n"
                "Choose the best next action."
            )
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda")

    # Generate next action
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    print(f"\n--- STEP {step} ---")
    print("Raw model output:", repr(response))

    match = re.fullmatch(r"\d+", response)

    if not match:
        status = "parse_error"
        trace.append({
            "step": step,
            "error": "parse_error",
            "raw_response": response,
            "available_actions": actions
        })
        print("PARSING ERROR")
        break

    index = int(response) - 1

    if not 0 <= index < len(actions):
        status = "invalid_action_index"
        trace.append({
            "step": step,
            "error": status,
            "raw_response": response,
            "available_actions": actions
        })
        print("INVALID ACTION INDEX")
        break

    chosen_action = actions[index]
    print("Action:", chosen_action)

    # Execute in ALFWorld
    try:
        next_obs, scores, dones, next_info = env.step(
            [chosen_action]
        )
    except Exception as exc:
        status = "environment_error"
        trace.append({
            "step": step,
            "action": chosen_action,
            "error": status,
            "details": repr(exc)
        })
        print("ENVIRONMENT ERROR:", repr(exc))
        break

    new_observation = next_obs[0]
    done = bool(dones[0])
    won = bool(next_info["won"][0])

    print("Observation:", new_observation)
    print("Success:", won)

    trace.append({
        "step": step,
        "input_observation": observation,
        "available_actions": actions,
        "raw_model_output": response,
        "action": chosen_action,
        "output_observation": new_observation,
        "score": float(scores[0]),
        "done": done,
        "won": won
    })

    history.append({
        "action": chosen_action,
        "observation": new_observation
    })
    # Stop if the agent repeats the same two-action pattern
    recent_actions = [
        item["action"] for item in trace[-6:]
    ]

    if done:
        status = "success" if won else "task_failed"
        break

    if (
        len(recent_actions) == 6
        and recent_actions[:2] == recent_actions[2:4]
        and recent_actions[2:4] == recent_actions[4:6]
    ):
        status = "repeated_action_loop"
        print("Agent stuck in a repeated action loop.")
        break
    observation = new_observation
    info = next_info

    if done:
        status = "success" if won else "task_failed"
        break

# Save debugging data
result = {
    "task": task,
    "initial_observation": initial_observation,
    "status": status,
    "steps": len(trace),
    "trace": trace
}

with open("/content/alfworld_trace.json", "w") as f:
    json.dump(result, f, indent=2)

print("\n" + "=" * 60)
print("FINAL STATUS:", status)
print("TOTAL STEPS:", len(trace))
print("TRACE SAVED: /content/alfworld_trace.json")
print("=" * 60)

TASK: cool some apple and put it in garbagecan.

--- STEP 1 ---
Raw model output: '31'
Action: go to fridge 1
Observation: You arrive at fridge 1. The fridge 1 is closed.
Success: False

--- STEP 2 ---
Raw model output: '33'
Action: go to microwave 1
Observation: You arrive at microwave 1. The microwave 1 is closed.
Success: False

--- STEP 3 ---
Raw model output: '32'
Action: go to fridge 1
Observation: You arrive at fridge 1. The fridge 1 is closed.
Success: False

--- STEP 4 ---
Raw model output: '43'
Action: open fridge 1
Observation: You open the fridge 1. The fridge 1 is open. In it, you see a lettuce 1, and a tomato 1.
Success: False

--- STEP 5 ---
Raw model output: '1'
Action: close fridge 1
Observation: You close the fridge 1.
Success: False

--- STEP 6 ---
Raw model output: '1'
Action: examine fridge 1
Observation: The fridge 1 is closed.
Success: False

--- STEP 7 ---
Raw model output: '33'
Action: go to microwave 1
Observation: You arrive at microwave 1. The microwave 1 is

AGENT REACHED MAX STEPS, DID NOT SUCCEED:
continuing to testing if they could go through successful sequence of steps to pass the benchmark


In [ ]:
# Reset the environment
obs, info = env.reset()

print("TASK:")
print(obs[0].split("Your task is to:")[-1].strip())

# Manually specified actions for this ONE debugging task
test_actions = [
    "go to garbagecan 1",
    "take apple 1 from garbagecan 1",
    "go to fridge 1",
    "cool apple 1 with fridge 1",
    "go to garbagecan 1",
    "move apple 1 to garbagecan 1"
]

for step, action in enumerate(test_actions, start=1):

    available = list(info["admissible_commands"][0])

    # Verify action is actually executable
    if action not in available:
        print(f"\nSTEP {step}: ACTION NOT AVAILABLE")
        print("Attempted:", action)
        print("Relevant actions:")
        print([
            a for a in available
            if "apple" in a or "fridge" in a
        ])
        break

    obs, scores, dones, info = env.step([action])

    print(f"\nSTEP {step}: {action}")
    print("Observation:", obs[0])
    print("Success:", info["won"][0])

    if dones[0]:
        break

print("\nFINAL SUCCESS:", info["won"][0])

TASK:
cool some apple and put it in garbagecan.

STEP 1: go to garbagecan 1
Observation: You arrive at garbagecan 1. On the garbagecan 1, you see a apple 1, and a soapbottle 1.
Success: False

STEP 2: take apple 1 from garbagecan 1
Observation: You pick up the apple 1 from the garbagecan 1.
Success: False

STEP 3: go to fridge 1
Observation: You arrive at fridge 1. The fridge 1 is closed.
Success: False

STEP 4: cool apple 1 with fridge 1
Observation: You cool the apple 1 using the fridge 1.
Success: False

STEP 5: go to garbagecan 1
Observation: You arrive at garbagecan 1. On the garbagecan 1, you see a soapbottle 1.
Success: False

STEP 6: move apple 1 to garbagecan 1
Observation: You move the apple 1 to the garbagecan 1.
Success: True

FINAL SUCCESS: True


OBSERVING OTHER HARNESS IMPLEMENTATIONS

In [ ]:
import json
import urllib.request

url = (
    "https://raw.githubusercontent.com/ysymyth/"
    "ReAct/master/prompts/alfworld_3prompts.json"
)

with urllib.request.urlopen(url) as response:
    react_prompts = json.load(response)

# Show an example for a cooling task
print(react_prompts["react_cool_0"])

You are in the middle of a room. Looking quickly around you, you see a cabinet 16, a cabinet 15, a cabinet 14, a cabinet 13, a cabinet 12, a cabinet 11, a cabinet 10, a cabinet 9, a cabinet 8, a cabinet 7, a cabinet 6, a cabinet 5, a cabinet 4, a cabinet 3, a cabinet 2, a cabinet 1, a coffeemachine 1, a countertop 2, a countertop 1, a diningtable 1, a drawer 5, a drawer 4, a drawer 3, a drawer 2, a drawer 1, a fridge 1, a garbagecan 1, a microwave 1, a safe 1, a sinkbasin 1, a stoveburner 4, a stoveburner 3, a stoveburner 2, a stoveburner 1, and a toaster 1.
Your task is to: cool some pan and put it in stoveburner.
> think: To solve the task, I need to find and take a pan, then cool it with fridge, then put it in stoveburner.
OK.
> think: First I need to find a pan. An pan is more likely to appear in stoveburner (1-4), sinkbasin (1), diningtable (1), countertop (1-2), cabinet (1-16), drawer (1-5). I can check one by one, starting with stoveburner 1.
OK.
>go to stoveburner 1
On the stov

In [ ]:
import torch
import json
import urllib.request

# Load the authors' demonstrations if not already loaded
if "react_prompts" not in globals():
    url = (
        "https://raw.githubusercontent.com/ysymyth/"
        "ReAct/master/prompts/alfworld_3prompts.json"
    )
    with urllib.request.urlopen(url) as response:
        react_prompts = json.load(response)

# Reset to our debugging task
obs, info = env.reset()
observation = obs[0].split("\n\n", 1)[-1]

# Use the original two cooling-task examples
examples = (
    react_prompts["react_cool_1"]
    + "\n"
    + react_prompts["react_cool_0"]
)

transcript = (
    "Interact with a household to solve a task. "
    "Here are two examples.\n"
    + examples
    + "\nHere is the task.\n"
    + observation
    + "\n>"
)

trace = []
status = "max_turns"
MAX_TURNS = 15

model.eval()

print("TASK:")
print(observation.split("Your task is to:")[-1].strip())

for step in range(1, MAX_TURNS + 1):

    messages = [
        {
            "role": "system",
            "content": (
                "You are an ALFWorld agent. "
                "Follow the format of the examples. "
                "On each turn, output exactly one line: "
                "either 'think: ...' or one executable "
                "ALFWorld action. Do not invent observations."
            )
        },
        {
            "role": "user",
            "content": transcript
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=90,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    raw = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    lines = [line.strip() for line in raw.splitlines()
             if line.strip()]
    action = lines[0].lstrip(">").strip() if lines else ""

    print(f"\n--- TURN {step} ---")
    print("Model:", action)

    record = {
        "turn": step,
        "raw_output": raw,
        "action": action
    }

    if action.lower().startswith("think:"):
        # Internal reasoning: do not execute in environment
        feedback = "OK."
        record["type"] = "thought"
        done = False

    elif action in info["admissible_commands"][0]:
        # Execute the model's chosen action
        next_obs, scores, dones, next_info = env.step([action])
        feedback = next_obs[0]
        done = bool(dones[0])
        won = bool(next_info["won"][0])

        record["type"] = "environment_action"
        record["won"] = won
        record["done"] = done

        info = next_info

    else:
        # Log invalid actions instead of silently correcting them
        record["type"] = "invalid_action"
        record["admissible_actions"] = list(
            info["admissible_commands"][0]
        )
        feedback = "INVALID ACTION"
        done = False
        status = "invalid_action"
        trace.append(record)
        print(feedback)
        break

    print("Feedback:", feedback)

    record["feedback"] = feedback
    trace.append(record)

    transcript += f" {action}\n{feedback}\n>"

    if done:
        status = "success" if won else "task_failed"
        break

with open("/content/react_pilot_trace.json", "w") as f:
    json.dump({
        "status": status,
        "trace": trace
    }, f, indent=2)

print("\nFINAL STATUS:", status)
print("TURNS:", len(trace))

TASK:
cool some apple and put it in garbagecan.

--- TURN 1 ---
Model: think: To solve the task, I need to find and take an apple, then cool it with the fridge, and finally put it in the garbagecan.
Feedback: OK.

--- TURN 2 ---
Model: think: First, I need to locate an apple. Apples are more likely to be found on countertops (1-2), dining table (1), cabinets (1-21), drawers (1-5), and fridge (1). I will start checking from countertop 1.
Feedback: OK.

--- TURN 3 ---
Model: go to countertop 1
Feedback: You arrive at countertop 1. On the countertop 1, you see a bowl 1, a bread 1, a knife 2, and a pan 2.

--- TURN 4 ---
Model: go to countertop 2
Feedback: You arrive at countertop 2. On the countertop 2, you see a egg 2, a fork 2, a potato 3, a potato 2, a saltshaker 2, and a spoon 3.

--- TURN 5 ---
Model: go to diningtable 1
Feedback: You arrive at diningtable 1. On the diningtable 1, you see a butterknife 2, a butterknife 1, a dishsponge 1, a knife 1, a mug 1, a potato 1, a saltshaker 1

In [ ]:
action = "go to cabinet 2"

available = list(info["admissible_commands"][0])

print("Listed as admissible:", action in available)

print("\nCabinet navigation commands:")
for cmd in available:
    if cmd.startswith("go to cabinet"):
        print(repr(cmd))

print("\nTrying the action directly:")
obs_test, scores_test, dones_test, info_test = env.step([action])

print("Environment response:", obs_test[0])

Listed as admissible: False

Cabinet navigation commands:
'go to cabinet 10'
'go to cabinet 11'
'go to cabinet 12'
'go to cabinet 13'
'go to cabinet 14'
'go to cabinet 15'
'go to cabinet 16'
'go to cabinet 17'
'go to cabinet 18'
'go to cabinet 19'
'go to cabinet 20'
'go to cabinet 21'
'go to cabinet 3'
'go to cabinet 4'
'go to cabinet 5'
'go to cabinet 6'
'go to cabinet 7'
'go to cabinet 8'
'go to cabinet 9'

Trying the action directly:
Environment response: Nothing happens.


In [ ]:
obs, scores, dones, info = env.step(["look"])

print("OBSERVATION:")
print(obs[0])

print("\nACTIONS INVOLVING CABINET 2:")
for action in info["admissible_commands"][0]:
    if "cabinet 2" in action:
        print(action)

OBSERVATION:
You are facing the cabinet 2, and cabinet 1. Next to it, you see nothing.

ACTIONS INVOLVING CABINET 2:
examine cabinet 2
go to cabinet 20
go to cabinet 21


In [ ]:
import torch
import json
import urllib.request

# ==========================================
# 1. CONFIGURATION
# ==========================================

MAX_TURNS = 40
MAX_ACTIONS = 30

# Load official ReAct demonstrations
if "react_prompts" not in globals():
    url = (
        "https://raw.githubusercontent.com/ysymyth/"
        "ReAct/master/prompts/alfworld_3prompts.json"
    )
    with urllib.request.urlopen(url) as response:
        react_prompts = json.load(response)

# Reset ALFWorld
obs, info = env.reset()
observation = obs[0].split("\n\n", 1)[-1]

# Use the two cooling-task demonstrations
examples = (
    react_prompts["react_cool_1"]
    + "\n"
    + react_prompts["react_cool_0"]
)

transcript = (
    "Interact with a household to solve a task. "
    "Here are two examples.\n"
    + examples
    + "\nHere is the task.\n"
    + observation
    + "\n>"
)

trace = []
status = "max_turns"
action_count = 0
thought_count = 0
no_op_count = 0

model.eval()

print("=" * 60)
print("TASK:")
print(observation.split("Your task is to:")[-1].strip())
print("=" * 60)

# ==========================================
# 2. AGENT EXECUTION LOOP
# ==========================================

for step in range(1, MAX_TURNS + 1):

    messages = [
        {
            "role": "system",
            "content": (
                "You are an ALFWorld agent. "
                "Follow the format of the examples. "
                "On each turn, output exactly one line: "
                "either 'think: ...' or one executable "
                "ALFWorld action. Do not invent observations."
                "Before searching, identify possible object locations. "
                "Prioritize a diverse set of plausible locations rather than "
                "exhaustively searching every cabinet or drawer. "
                "Consider all receptacles, including the task destination. "
                "Remember which locations have been checked and avoid "
                "unnecessary revisits. "
                "If a search strategy is unsuccessful, reconsider your priorities."
            )
        },
        {
            "role": "user",
            "content": transcript
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to("cuda")

    # Model inference
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=90,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    raw = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    lines = [
        line.strip()
        for line in raw.splitlines()
        if line.strip()
    ]

    action = (
        lines[0].lstrip(">").strip()
        if lines else ""
    )

    print(f"\n--- TURN {step} ---")
    print("Model:", action)

    record = {
        "turn": step,
        "raw_output": raw,
        "action": action
    }

    # ======================================
    # 3. INTERNAL REASONING
    # ======================================

    if action.lower().startswith("think:"):

        feedback = "OK."
        done = False

        thought_count += 1

        record["type"] = "thought"
        record["feedback"] = feedback

    # ======================================
    # 4. ENVIRONMENT ACTION
    # ======================================

    elif action:

        available = list(
            info["admissible_commands"][0]
        )

        was_admissible = action in available

        record["type"] = "environment_action"
        record["admissible"] = was_admissible
        record["available_actions"] = available

        try:
            next_obs, scores, dones, next_info = env.step(
                [action]
            )

            feedback = next_obs[0]
            done = bool(dones[0])
            won = bool(next_info["won"][0])

            action_count += 1

            # Detect actions with no apparent effect
            no_op = (
                feedback.strip().lower()
                == "nothing happens."
            )

            if no_op:
                no_op_count += 1

            record["feedback"] = feedback
            record["done"] = done
            record["won"] = won
            record["score"] = float(scores[0])
            record["no_op"] = no_op

            # Update environment information
            info = next_info

            if not was_admissible:
                print(
                    "WARNING: Action not listed as admissible"
                )

            if no_op:
                print(
                    "WARNING: Action had no apparent effect"
                )

        except Exception as e:

            record["type"] = "environment_error"
            record["error"] = repr(e)

            trace.append(record)
            status = "environment_error"

            print("ENVIRONMENT ERROR:", e)
            break

    # ======================================
    # 5. EMPTY MODEL OUTPUT
    # ======================================

    else:

        record["type"] = "empty_output"
        trace.append(record)

        status = "empty_output"

        print("ERROR: Model returned no action")
        break

    # ======================================
    # 6. LOG AND UPDATE TRANSCRIPT
    # ======================================

    print("Feedback:", feedback)

    record["feedback"] = feedback
    trace.append(record)

    transcript += f" {action}\n{feedback}\n>"

    # Task completed or terminated
    if done:
        status = "success" if won else "task_failed"
        break

    # Environment action budget reached
    if action_count >= MAX_ACTIONS:
        status = "max_actions"
        break


# ==========================================
# 7. SAVE RESULTS
# ==========================================

result = {
    "task": observation.split(
        "Your task is to:"
    )[-1].strip(),
    "status": status,
    "total_turns": len(trace),
    "environment_actions": action_count,
    "reasoning_steps": thought_count,
    "no_op_actions": no_op_count,
    "trace": trace
}

output_path = "/content/react_pilot_trace.json"

with open(output_path, "w") as f:
    json.dump(result, f, indent=2)

# ==========================================
# 8. FINAL SUMMARY
# ==========================================

print("\n" + "=" * 60)
print("FINAL STATUS:", status)
print("TOTAL TURNS:", len(trace))
print("ENVIRONMENT ACTIONS:", action_count)
print("REASONING STEPS:", thought_count)
print("NO-OP ACTIONS:", no_op_count)
print("TRACE SAVED:", output_path)
print("=" * 60)

TASK:
cool some apple and put it in garbagecan.

--- TURN 1 ---
Model: think: To solve the task, I need to find and take an apple, then cool it with the fridge, and finally put it in the garbagecan. Apples are more likely to be found in the countertop (1-2), diningtable (1), fridge (1), and possibly cabinets (1-21). I will start checking from the countertop.
Feedback: OK.

--- TURN 2 ---
Model: go to countertop 1
Feedback: You arrive at countertop 1. On the countertop 1, you see a bowl 1, a bread 1, a knife 2, and a pan 2.

--- TURN 3 ---
Model: go to countertop 2
Feedback: You arrive at countertop 2. On the countertop 2, you see a egg 2, a fork 2, a potato 3, a potato 2, a saltshaker 2, and a spoon 3.

--- TURN 4 ---
Model: go to diningtable 1
Feedback: You arrive at diningtable 1. On the diningtable 1, you see a butterknife 2, a butterknife 1, a dishsponge 1, a knife 1, a mug 1, a potato 1, a saltshaker 1, a spoon 2, a spoon 1, a tomato 3, and a tomato 2.

--- TURN 5 ---
Model: go to